In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("prueba_cluster_pyspark").getOrCreate()

print("Versión de Spark:", spark.version)
print("Master:", spark.sparkContext.master)

In [ ]:
import socket

print("Filas:", spark.range(0, 1_000_000, numPartitions=6).count())

nodos = (spark.sparkContext
         .parallelize(range(6), 6)
         .map(lambda _: socket.gethostname())
         .distinct()
         .collect())
print("Nodos que han ejecutado tareas:", sorted(nodos))

In [ ]:
from pyspark.sql import functions as F

ventas = spark.createDataFrame(
    [("Madrid", "2026-10", 1250.40), ("Madrid", "2026-11", 980.10),
     ("Valencia", "2026-10", 640.00), ("Valencia", "2026-11", 715.55),
     ("Sevilla", "2026-11", 1120.00)],
    "tienda STRING, mes STRING, importe DOUBLE",
)

resumen = (
    ventas
    .groupBy("tienda")
    .agg(F.count("*").alias("meses"), F.round(F.sum("importe"), 2).alias("total"))
    .orderBy("tienda")
)
resumen.show()

resumen.write.mode("overwrite").parquet("/opt/datos/resumen_ventas")
print("Guardado en /opt/datos/resumen_ventas")

In [ ]:
spark.stop()